# Week 1 — Bulk Parsing Pipeline

**Goal:** Parse all personal replays, apply filters, and produce a flat CSV ready for feature engineering.

**Output:** `data/parsed_replays.csv` — one row per **player** per game (~388 rows from ~194 games, 70+ feature columns)

**Pipeline (3 phases):**
1. Header scan — cheaply filter 3400+ non-Arabia / unrated / team-game files, now parallelised (~2 min)
2. Full parse — run `parse_replay()` on ~215 qualifying candidates via `ThreadPoolExecutor` (~67s)
3. Flatten — convert nested player_stats dicts to flat rows, build DataFrame, save CSV

**Filters applied:**
- Rated game (`de['rated'] == True`)
- Arabia map (`rms_map_id == 9`)
- 1v1 (`num_players == 2`)
- Duration > 8 minutes (removes disconnects and early resigns)

**Performance:** Both phases use `ThreadPoolExecutor(12)`. Header parse is file I/O + zlib — releases the GIL, so threads run genuinely in parallel. Phase 1 drops from ~23 min to ~2 min.

## Setup

In [1]:
import os
import struct
import logging
import time
import pandas as pd
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from mgz.fast import header as fast_header, operation, Operation, meta
from mgz.fast.header import parse as fast_header_parse
from mgz.fast.enums import Action

# ── Constants ─────────────────────────────────────────────────────────────────
ARABIA_MAP_ID          = 9
RM_1V1_LEADERBOARD_ID  = 3
MIN_DURATION_MIN       = 8
N_THREADS              = 12

# Your in-game name — used to derive is_me when profile_id is unavailable
MY_PLAYER_NAME = {'TheRealRuClEsHe', 'b\'TheRealRuClEsHe\''} # include both raw and bytestring versions to be safe

REPLAY_DIR  = Path('C:/Users/liher/Games/Age of Empires 2 DE/76561198151543542/savegame')
OUTPUT_PATH = Path('../data/parsed_replays.csv')
LOG_PATH    = Path('../data/parsing_failures.log')

LOG_PATH.parent.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    filename=str(LOG_PATH),
    level=logging.ERROR,
    format='%(asctime)s — %(message)s'
)

# ── Tech / Building ID maps (must match 01_exploration_new.ipynb) ─────────────
VILLAGER_IDS = {83, 293}   # 83 = male villager, 293 = female villager (Aztecs + civ-dependent)

TRACKED_BUILDINGS = {
    12:  'first_barracks_min',
    101: 'first_stable_min',
    87:  'first_archery_min',
    103: 'blacksmith_min',
}
WALL_IDS   = {72, 117, 155}  # Palisade, Stone, Fortified
MARKET_IDS = {84, 116, 137}  # Feudal, Castle, Imperial age market

TRACKED_TECHS = {
    101: 'feudal_min', 102: 'castle_min', 103: 'imperial_min',
    211: 'double_bit_axe_min',   212: 'bow_saw_min',            221: 'two_man_saw_min',
    14:  'horse_collar_min',     13:  'heavy_plow_min',         12:  'crop_rotation_min',
    55:  'gold_mining_min',      182: 'gold_shafting_mining_min',
    278: 'stone_mining_min',     279: 'stone_shafting_mining_min',
    213: 'wheelbarrow_min',      249: 'hand_cart_min',
    8:   'town_watch_min',       280: 'town_patrol_min',        22: 'loom_min',
    67:  'forging_min',          68:  'iron_casting_min',       75: 'blast_furnace_min',
    74:  'scale_mail_armor_min', 76:  'chain_mail_armor_min',   77: 'plate_mail_armor_min',
    81:  'scale_barding_armor_min', 82: 'chain_barding_armor_min', 80: 'plate_barding_armor_min',
    199: 'fletching_min',        200: 'bodkin_arrow_min',       201: 'bracer_min',
    215: 'padded_archer_armor_min', 218: 'leather_archer_armor_min', 219: 'ring_archer_armor_min',
    435: 'bloodlines_min',       39:  'husbandry_min',
    437: 'thumb_ring_min',       436: 'parthian_tactics_min',
    602: 'arson_min',            875: 'gambeson_min',           216: 'squires_min', 47: 'chemistry_min',
}

print('Imports OK')

Imports OK


## Section 1 — Find Replay Files

In [2]:
replay_files = sorted(REPLAY_DIR.glob('*.aoe2record'))
print(f'Found {len(replay_files)} total replay files')

Found 3649 total replay files


## Section 2 — Parser Functions

- `parse_header()` — header block only (~0.05s/file). Returns map_id, rated, num_players for filtering. Never touches the body.
- `parse_replay()` — full parse (header + body). 70+ features per player, plus result and elo. Only runs on the ~215 candidates that passed the header filter.

In [3]:
def parse_header(filepath):
    """Fast header-only parse. Returns (map_id, rated, num_players)."""
    with open(filepath, 'rb') as f:
        h       = fast_header_parse(f)
        de      = h.get('de') or {}
        players = [p for p in (h.get('players') or []) if p and p.get('type') == 1]
        return de.get('rms_map_id'), de.get('rated', False), len(players)

In [4]:
def parse_replay(filepath):
    """Full parse of one .aoe2record file.
    Returns a dict with game-level metadata and per-player feature dicts.
    result (0=loss, 1=win) and elo are derived inside this function.
    """
    filepath = str(filepath)

    with open(filepath, 'rb') as f:

        # ── Header ────────────────────────────────────────────────────────────
        h          = fast_header_parse(f)
        de         = h.get('de') or {}
        map_id     = de.get('rms_map_id')
        rated      = de.get('rated', False)

        header_players = {p['number']: p for p in (h.get('players') or [])
                          if p and p.get('type') == 1}
        de_players     = {p['number']: p for p in (de.get('players') or [])
                          if p.get('number', -1) >= 1}

        # ── Feature init ──────────────────────────────────────────────────────
        player_stats = defaultdict(lambda: {
            'villagers_dark_age': 0, 'villagers_feudal_age': 0,
            'villagers_castle_age': 0, 'villagers_imperial_age': 0,
            'mil_trained_dark_age': 0, 'mil_trained_feudal_age': 0,
            'mil_trained_castle_age': 0, 'mil_trained_imperial_age': 0,
            'apm_dark_age': 0, 'apm_feudal_age': 0,
            'apm_castle_age': 0, 'apm_imperial_age': 0,
            'gather_point_count': 0, 'wall_count': 0,
            'move_count': 0, 'order_count': 0, 'stance_count': 0, 'ungarrison_count': 0,
            'first_wall_min': None, 'resign_time_min': None, 'resigned': False,
            'market_min': None, 'market_count': 0,
            **{v: None for v in TRACKED_BUILDINGS.values()},
            **{v: None for v in TRACKED_TECHS.values()},
            'thresholds': {
                'feudal_complete':   float('inf'),
                'castle_complete':   float('inf'),
                'imperial_complete': float('inf'),
            },
            'military_buildings_placed': defaultdict(int),
        })

        leaderboard_ratings = {}
        game_time_ms        = 0
        world_time_ms       = None

        # ── Body scan ─────────────────────────────────────────────────────────
        meta(f)
        eof = os.fstat(f.fileno()).st_size

        while f.tell() < eof:
            try:
                op_type, payload = operation(f)
            except EOFError:
                break

            if op_type == Operation.SYNC:
                game_time_ms += payload[0]
                continue

            if op_type == Operation.ACTION:
                action_type, ap = payload
                pid = ap.get('player_id')
                if not pid:
                    continue

                t   = round(game_time_ms / 60000, 2)
                s   = player_stats[pid]
                thr = s['thresholds']
                act = action_type.name if hasattr(action_type, 'name') else str(action_type)

                if   t < thr['feudal_complete']:   phase = 'dark_age'
                elif t < thr['castle_complete']:   phase = 'feudal_age'
                elif t < thr['imperial_complete']: phase = 'castle_age'
                else:                              phase = 'imperial_age'

                if act == 'RESIGN':
                    if not s['resigned']:
                        s['resigned']        = True
                        s['resign_time_min'] = t

                elif act == 'RESEARCH':
                    tech = ap.get('technology_id')
                    if tech == 101:   thr['feudal_complete']   = t + 2.16
                    elif tech == 102: thr['castle_complete']   = t + 2.66
                    elif tech == 103: thr['imperial_complete'] = t + 3.16
                    if tech in TRACKED_TECHS and s[TRACKED_TECHS[tech]] is None:
                        s[TRACKED_TECHS[tech]] = t

                elif act in ('BUILD', 'WALL'):
                    bid = ap.get('building_id')
                    if bid in TRACKED_BUILDINGS:
                        s['military_buildings_placed'][bid] += 1
                        feat = TRACKED_BUILDINGS[bid]
                        if s[feat] is None:
                            s[feat] = t
                    elif bid in WALL_IDS:
                        s['wall_count'] += 1
                        if s['first_wall_min'] is None:
                            s['first_wall_min'] = t
                    elif bid in MARKET_IDS:
                        s['market_count'] += 1
                        if s['market_min'] is None:
                            s['market_min'] = t

                if act in ('MOVE', 'ORDER', 'STANCE', 'BUILD', 'RESEARCH',
                           'DE_QUEUE', 'GATHER_POINT', 'DE_TRANSFORM'):
                    s[f'apm_{phase}'] += 1

                    if act == 'DE_QUEUE':
                        uid    = ap.get('unit_id')
                        amount = ap.get('amount', 1) or 1
                        if uid in VILLAGER_IDS: s[f'villagers_{phase}'] += amount
                        else:                  s[f'mil_trained_{phase}'] += amount
                    elif act == 'GATHER_POINT': s['gather_point_count'] += 1
                    elif act == 'MOVE':         s['move_count']         += 1
                    elif act == 'ORDER':        s['order_count']        += 1
                    elif act == 'STANCE':       s['stance_count']       += 1
                    elif act == 'UNGARRISON':   s['ungarrison_count']   += 1

            elif op_type == Operation.POSTGAME:
                world_time_ms = payload.get('world_time')
                for lb in (payload.get('leaderboards') or []):
                    if lb.get('id') == RM_1V1_LEADERBOARD_ID:
                        for entry in (lb.get('players') or []):
                            leaderboard_ratings[entry['number']] = entry['rating']
                break

    duration_min     = round((world_time_ms or game_time_ms) / 60000, 2)
    num_real_players = len(header_players)

    # result: resigned = loss, other = win
    losers = {pid for pid, s in player_stats.items() if s['resigned']}
    # elo: POSTGAME slots are 0-indexed; sorted player_ids are 1-indexed.
    # Assignment is approximate — see ISSUE-003. Acceptable for 200-wide cohort bands.
    sorted_pids    = sorted(player_stats.keys())
    elo_assignment = {pid: leaderboard_ratings.get(i) for i, pid in enumerate(sorted_pids)}

    for pid, s in player_stats.items():
        s['result'] = 0 if pid in losers else 1
        s['elo']    = elo_assignment.get(pid)

    return {
        'filepath':     filepath,
        'map_id':       map_id,
        'rated':        rated,
        'num_players':  num_real_players,
        'duration_min': duration_min,
        'de_players':   de_players,
        'player_stats': dict(player_stats),
    }

In [5]:
BUILDING_COUNT_MAP = {
    12:  'barracks_count',
    101: 'stable_count',
    87:  'archery_count',
    103: 'blacksmith_count',
}

def flatten_player_row(game, player_num):
    """Convert one player's stats dict to a flat row dict for a DataFrame."""
    s  = dict(game['player_stats'][player_num])
    dp = game['de_players'].get(player_num, {})

    profile_id  = dp.get('profile_id')
    player_name = dp.get('name', '')
    civ_id      = dp.get('civilization_id') or dp.get('civ_id')
    is_me       = (player_name in MY_PLAYER_NAME)

    mbp             = dict(s.pop('military_buildings_placed', {}))
    market_count    = s.pop('market_count', 0)
    building_counts = {name: mbp.get(bid, 0) for bid, name in BUILDING_COUNT_MAP.items()}
    building_counts['market_count'] = market_count

    s.pop('thresholds', None)
    s.pop('resigned', None)        # encoded in result
    s.pop('resign_time_min', None) # post-hoc — leaks result

    return {
        'filepath':        game['filepath'],
        'map_id':          game['map_id'],
        'rated':           game['rated'],
        'num_players':     game['num_players'],
        'duration_min':    game['duration_min'],
        'player_num':      player_num,
        'profile_id':      profile_id,
        'player_name':     player_name,
        'civilization_id': civ_id,
        'is_me':           is_me,
        **s,
        **building_counts,
    }

## Section 3 — Bulk Parse Pipeline

**Phase 1 shortcut:** if `parsed_replays.csv` already exists, skip Phase 1 and load filepaths from it.
Set `SKIP_PHASE1 = False` or delete the old CSV to force a full rescan.

In [6]:
SKIP_PHASE1 = OUTPUT_PATH.exists()

if SKIP_PHASE1:
    df_prev    = pd.read_csv(OUTPUT_PATH)
    print("Actual columns found:", df_prev.columns.tolist()) # Debugging line
    candidates = df_prev['filepath'].unique().tolist()
    print(f'Shortcut: loaded {len(candidates)} candidate filepaths from existing {OUTPUT_PATH}')
    print('Phase 1 skipped. Delete parsed_replays.csv and rerun to force a full rescan.')
else:
    candidates = None
    print('No existing CSV — will run full Phase 1 header scan.')

No existing CSV — will run full Phase 1 header scan.


In [7]:
if not SKIP_PHASE1:
    # Header parse = file I/O + zlib decompression — both release the GIL.
    # 12 threads gives ~12x speedup: ~23 min → ~2 min across 3600+ files.
    def safe_header(fp):
        try:
            map_id, rated, num_players = parse_header(fp)
            passed = (map_id == ARABIA_MAP_ID and rated and num_players == 2)
            return str(fp), passed, None
        except Exception as e:
            logging.error(f'HEADER FAILED: {fp} | {type(e).__name__}: {e}')
            return str(fp), False, str(e)

    print(f'Phase 1 — header scan ({len(replay_files)} files, {N_THREADS} threads)...')
    candidates     = []
    skipped_header = 0
    header_errors  = []
    t1 = time.time()

    with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
        futures = {executor.submit(safe_header, fp): fp for fp in replay_files}
        done = 0
        for future in as_completed(futures):
            fp_str, passed, err = future.result()
            done += 1
            if err:
                header_errors.append(fp_str)
            elif passed:
                candidates.append(fp_str)
            else:
                skipped_header += 1
            if done % 500 == 0 or done == len(replay_files):
                print(f'  {done}/{len(replay_files)} scanned...')

    print(f'  Done in {round(time.time()-t1, 1)}s')
    print(f'  Candidates: {len(candidates)}  |  Skipped: {skipped_header}  |  Errors: {len(header_errors)}')
else:
    print('Phase 1 skipped (shortcut active).')

Phase 1 — header scan (3649 files, 12 threads)...
  500/3649 scanned...
  1000/3649 scanned...
  1500/3649 scanned...
  2000/3649 scanned...
  2500/3649 scanned...
  3000/3649 scanned...
  3500/3649 scanned...
  3649/3649 scanned...
  Done in 1362.8s
  Candidates: 215  |  Skipped: 3429  |  Errors: 5


In [8]:
# Phase 2 — Full parse on ~215 candidates
def safe_parse(filepath_str):
    try:
        return parse_replay(filepath_str)
    except Exception as e:
        logging.error(f'BODY FAILED: {filepath_str} | {type(e).__name__}: {e}')
        return None

print(f'Phase 2 — full parse ({len(candidates)} files, {N_THREADS} threads)...')
t2 = time.time()
raw_games   = []
body_errors = 0

with ThreadPoolExecutor(max_workers=N_THREADS) as executor:
    futures = {executor.submit(safe_parse, fp): fp for fp in candidates}
    done = 0
    for future in as_completed(futures):
        result = future.result()
        done  += 1
        if result is None:
            body_errors += 1
        else:
            raw_games.append(result)
        if done % 25 == 0 or done == len(candidates):
            print(f'  {done}/{len(candidates)} complete')

print(f'  Done in {round(time.time()-t2, 1)}s')
print(f'  Parsed OK: {len(raw_games)}  |  Errors: {body_errors}')

Phase 2 — full parse (215 files, 12 threads)...
  25/215 complete
  50/215 complete
  75/215 complete
  100/215 complete
  125/215 complete
  150/215 complete
  175/215 complete
  200/215 complete
  215/215 complete
  Done in 304.4s
  Parsed OK: 215  |  Errors: 0


In [9]:
# Phase 3 — Duration filter + flatten to rows
print('Phase 3 — duration filter + flatten...')
rows             = []
skipped_duration = 0
skipped_no_resign = 0

for game in raw_games:
    if game['duration_min'] <= MIN_DURATION_MIN:
        skipped_duration += 1
        continue
    # Drop games where result is ambiguous (no resign = both labelled win)
    resign_count = sum(1 for s in game['player_stats'].values() if s['resigned']) 
    if resign_count != 1: # must be exactly 1 resign to have a clear winner/loser
        skipped_no_resign += 1
        continue 
    for player_num in game['player_stats'].keys():
        try:
            rows.append(flatten_player_row(game, player_num))
        except Exception as e:
            logging.error(f'FLATTEN FAILED: {game["filepath"]} player {player_num} | {e}')

print(f'  Games kept:             {len(raw_games) - skipped_duration}')
print(f'  Games skipped (<8 min):      {skipped_duration}')
print(f'  Games skipped (no resign):   {skipped_no_resign}')
print(f'  Total rows (players):   {len(rows)}')

Phase 3 — duration filter + flatten...
  Games kept:             194
  Games skipped (<8 min):      21
  Games skipped (no resign):   19
  Total rows (players):   350


## Section 4 — Build DataFrame & Save

In [10]:
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved → {OUTPUT_PATH}')
print(f'Shape:  {df.shape}  (rows=player-games, cols=features)')
print(f'\nColumns:')
for c in df.columns:
    print(f'  {c}')

Saved → ..\data\parsed_replays.csv
Shape:  (350, 82)  (rows=player-games, cols=features)

Columns:
  filepath
  map_id
  rated
  num_players
  duration_min
  player_num
  profile_id
  player_name
  civilization_id
  is_me
  villagers_dark_age
  villagers_feudal_age
  villagers_castle_age
  villagers_imperial_age
  mil_trained_dark_age
  mil_trained_feudal_age
  mil_trained_castle_age
  mil_trained_imperial_age
  apm_dark_age
  apm_feudal_age
  apm_castle_age
  apm_imperial_age
  gather_point_count
  wall_count
  move_count
  order_count
  stance_count
  ungarrison_count
  first_wall_min
  market_min
  first_barracks_min
  first_stable_min
  first_archery_min
  blacksmith_min
  feudal_min
  castle_min
  imperial_min
  double_bit_axe_min
  bow_saw_min
  two_man_saw_min
  horse_collar_min
  heavy_plow_min
  crop_rotation_min
  gold_mining_min
  gold_shafting_mining_min
  stone_mining_min
  stone_shafting_mining_min
  wheelbarrow_min
  hand_cart_min
  town_watch_min
  town_patrol_min
  loo

## Section 5 — Summary & Validation

In [11]:
total_kept   = len(raw_games) - skipped_duration
failure_rate = round(100 * body_errors / len(candidates), 1) if candidates else 0

print('=' * 55)
print('PIPELINE SUMMARY')
print('=' * 55)
print(f'  Total replay files scanned:  {len(replay_files)}')
print(f'  Arabia 1v1 rated candidates: {len(candidates)}')
print(f'  Games kept (>8 min):         {total_kept}')
print(f'  Player rows in CSV:          {len(df)}')
print(f'  Parse failure rate:          {failure_rate}%  (target: <20%)')
print(f'  My rows (is_me=True):        {df["is_me"].sum()}')
print(f'  Opponent rows:               {(~df["is_me"]).sum()}')

print('\nClass balance (result):')
print(df['result'].value_counts().to_string())

print('\nNull counts (top 20 by null rate):')
null_pct = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
print(null_pct[null_pct > 0].head(20).round(1).to_string())

print('\nElo distribution (is_me rows):')
print(df[df['is_me']]['elo'].describe().round(0).to_string())

PIPELINE SUMMARY
  Total replay files scanned:  3649
  Arabia 1v1 rated candidates: 215
  Games kept (>8 min):         194
  Player rows in CSV:          350
  Parse failure rate:          0.0%  (target: <20%)
  My rows (is_me=True):        0
  Opponent rows:               350

Class balance (result):
result
0    175
1    175

Null counts (top 20 by null rate):
squires_min                  100.0
two_man_saw_min               98.3
gambeson_min                  98.0
town_patrol_min               97.4
stone_shafting_mining_min     97.1
crop_rotation_min             96.3
leather_archer_armor_min      96.0
parthian_tactics_min          95.4
plate_mail_armor_min          92.0
gold_shafting_mining_min      91.7
arson_min                     90.9
ring_archer_armor_min         90.6
thumb_ring_min                89.1
blast_furnace_min             86.0
plate_barding_armor_min       84.3
stone_mining_min              83.1
chain_mail_armor_min          82.6
padded_archer_armor_min       82.0
chemis